### Middleware
<br>
Middleware provides a way to more tightly control what happends inside the agent, middle is useful for following:
<br>
1. tracking agent behaviour with logging, analytics, and debugging
<br>
2. transforming prompts, tool selection, and output formatting
<br>
3. adding retries, fallbacks, and early termination logic
<br>
4. applying rate limits, guardrailes

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")

Summarization middleware: it automatically suimmarizes conversation history when approaching token limits, preserving recent messages while compresing older context

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, AIMessage

agent = create_agent(
    model = 'groq:qwen/qwen3-32b',
    checkpointer= InMemorySaver(),
    middleware=[ SummarizationMiddleware(
        model = 'groq:qwen/qwen3-32b',  #we must use  lighter cost saving model for summarization
        trigger = ("messages",10), #when message token exceeds it triggers the summarization middleware for past messages
        keep = ("messages",4) #preserve recent top 4 messages

    )]
)

just creating a config user to test the agent

In [7]:
config = {"configurable":{"thread_id":"test-1"}}

In [9]:
questions = [
    "what is 1+1?",
    "what is 1+4?",
    "what is 1+7?",
    "what is 1+10?",
    "what is 1+3?",
    "what is 1+9?",
    "what is 1+6?"
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content = q)]}, config)
    print(f"Messages:{response}")
    print(f"Messages: {len(response['messages'])}")

Messages:{'messages': [HumanMessage(content='what is 1+1?', additional_kwargs={}, response_metadata={}, id='ad95fb8a-b65b-4a39-9fd4-38574cdb1ee6'), AIMessage(content='<think>\nOkay, the user is asking "what is 1+1?" Let me think. First, this is a basic arithmetic question. The answer is straightforward, but maybe there\'s more to it. They might be testing if I can do simple math or looking for a deeper explanation.\n\nHmm, in standard arithmetic, 1 plus 1 equals 2. That\'s the usual answer. But sometimes people use this question to introduce concepts like binary, where 1+1 is 10. Or maybe in different number systems or contexts like modulo arithmetic. But unless specified, the default is decimal.\n\nWait, the user didn\'t specify any context, so probably they just want the basic answer. But maybe I should mention other possibilities briefly? Like in binary, 1+1 is 10. Or in some programming languages, maybe 1+1 could be different if there\'s a trick, but that\'s unlikely. \n\nAlso, con